In [1]:
# uv add langchain-google-genai

In [2]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
#print(GOOGLE_API_KEY[:4])

In [3]:
# google-genai SDK 불러오기 (langchain-google-genai가 내부적으로 사용하는 SDK)
from google import genai

# API 키로 클라이언트 생성 (GOOGLE_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = genai.Client(api_key=GOOGLE_API_KEY)

# 계정에서 접근 가능한 전체 모델 목록을 순회
for m in client.models.list():
    # generateContent(텍스트 생성) 지원 모델만 필터링 (임베딩 전용 모델 등 제외)
    # supported_actions가 None일 경우를 대비해 빈 리스트로 대체
    if "generateContent" in (m.supported_actions or []):
        # "models/gemini-2.5-flash" -> "gemini-2.5-flash" (접두사 제거)
        # 이 이름을 ChatGoogleGenerativeAI(model=...)에 그대로 사용 가능
        print(m.name.removeprefix("models/"), "|", m.display_name)

gemini-2.5-flash | Gemini 2.5 Flash
gemini-2.5-pro | Gemini 2.5 Pro
gemini-2.5-flash-preview-tts | Gemini 2.5 Flash Preview TTS
gemini-2.5-pro-preview-tts | Gemini 2.5 Pro Preview TTS
gemma-4-26b-a4b-it | Gemma 4 26B A4B IT
gemma-4-31b-it | Gemma 4 31B IT
gemini-flash-latest | Gemini Flash Latest
gemini-flash-lite-latest | Gemini Flash-Lite Latest
gemini-pro-latest | Gemini Pro Latest
gemini-2.5-flash-lite | Gemini 2.5 Flash-Lite
gemini-2.5-flash-image | Nano Banana
gemini-3-flash-preview | Gemini 3 Flash Preview
gemini-3.1-pro-preview | Gemini 3.1 Pro Preview
gemini-3.1-pro-preview-customtools | Gemini 3.1 Pro Preview Custom Tools
gemini-3.1-flash-lite-preview | Gemini 3.1 Flash Lite Preview
gemini-3.1-flash-lite | Gemini 3.1 Flash Lite
gemini-3-pro-image-preview | Nano Banana Pro
gemini-3-pro-image | Nano Banana Pro
nano-banana-pro-preview | Nano Banana Pro
gemini-3.1-flash-image-preview | Nano Banana 2
gemini-3.1-flash-image | Nano Banana 2
gemini-3.1-flash-lite-image | Nano Banana 

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
    
# 사용할 Gemini 모델 — 이 노트북의 모든 예제에서 공통 사용 (모델 변경은 이곳만 수정)
GOOGLE_MODEL = "gemini-3.1-flash-lite"

# 모델 초기화
llm = ChatGoogleGenerativeAI(
    model=GOOGLE_MODEL,
    temperature=0.3    
)


In [5]:

# 프롬프트 설정
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 AI 전문가입니다."),
    ("human", "{topic}은(는) 무엇인가요?")
])

# 체인 실행
chain = prompt | llm
response = chain.invoke({"topic": "LangChain과 LangGraph"})

print(" Google Gemini Response:")
print(response)
print(response.content)

 Google Gemini Response:
content=[{'type': 'text', 'text': 'AI 전문가로서 **LangChain**과 **LangGraph**의 개념과 차이점, 그리고 이들이 왜 현대 AI 개발에서 중요한지 설명해 드리겠습니다.\n\n---\n\n### 1. LangChain (랭체인)이란?\n**LangChain**은 LLM(거대 언어 모델)을 기반으로 한 애플리케이션을 개발하기 위한 **오픈소스 프레임워크**입니다.\n\n*   **핵심 역할:** LLM은 그 자체로는 외부 데이터에 접근하거나, 복잡한 계산을 수행하거나, 특정 작업을 순차적으로 처리하는 데 한계가 있습니다. LangChain은 이러한 LLM을 외부 도구(데이터베이스, 검색 엔진, API 등)와 연결하여 **\'에이전트\'**처럼 동작하게 만드는 \'접착제\' 역할을 합니다.\n*   **주요 구성 요소:**\n    *   **Model I/O:** 다양한 LLM(OpenAI, Anthropic, Llama 등)을 쉽게 교체하고 관리할 수 있는 인터페이스.\n    *   **Retrieval (RAG):** 외부 문서를 검색하고 LLM에 전달하여 답변의 정확도를 높이는 기능.\n    *   **Chains:** 여러 작업을 순차적으로 연결하는 파이프라인.\n    *   **Memory:** 대화의 맥락을 기억하게 하는 기능.\n\n---\n\n### 2. LangGraph (랭그래프)란?\n**LangGraph**는 LangChain 생태계의 일부로, **LLM 기반의 에이전트를 \'그래프(Graph)\' 구조로 설계하고 실행하기 위한 라이브러리**입니다.\n\n*   **왜 필요한가?** 기존의 LangChain \'Chain\'은 주로 선형적(A → B → C)인 흐름에 강했습니다. 하지만 복잡한 AI 에이전트는 **순환(Loop)** 구조가 필요합니다. 예를 들어, "답변이 틀렸으면 다시 검색하고, 다시 생성하라"와 같은 반복 작업이 필요할 때 L

#### Gemini 모델별 특징

* gemini-1.5-flash: 빠른 응답, 일반적인 작업에 적합
* gemini-2.5-pro: 더 정확하고 복잡한 추론 작업
* gemini-pro-vision: 이미지 처리 및 멀티모달 작업

In [6]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 기본 모델: 위 셀에서 GOOGLE_MODEL 로 만든 llm 을 재사용합니다. (temperature=0.3)

print("=" * 50)
print("예제 1: 기본 대화형 챗봇")
print("=" * 50)

# 대화형 프롬프트
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친근하고 도움이 되는 AI 어시스턴트입니다."),
    ("human", "{user_input}")
])

chat_chain = chat_prompt | llm | StrOutputParser()
response1 = chat_chain.invoke({"user_input": "파이썬으로 리스트를 정렬하는 방법은?"})
print("응답:", response1)

print("\n" + "=" * 50)
print("예제 2: JSON 구조화 출력")
print("=" * 50)

json_prompt = PromptTemplate(
    template="""
다음 정보를 JSON 형태로 변환하세요:
{company_info}

형식: {{"name": "회사명", "year": "연도", "location": "위치"}}
""",
    input_variables=["company_info"]
)

json_chain = json_prompt | llm | StrOutputParser()
company_text = "네이버는 1999년에 설립된 한국의 IT 기업이며 본사는 경기도 성남에 있습니다."
response2 = json_chain.invoke({"company_info": company_text})
print("JSON 결과:", response2)


예제 1: 기본 대화형 챗봇
응답: 파이썬에서 리스트를 정렬하는 방법은 크게 두 가지가 있습니다. 상황에 맞춰 선택해서 사용해 보세요!

---

### 1. `sort()` 메서드 (원본 리스트를 직접 수정)
리스트 객체 자체를 정렬하고 싶을 때 사용합니다. 원본 데이터가 변경되며, 반환값은 `None`입니다.

```python
numbers = [3, 1, 4, 1, 5, 9]
numbers.sort()
print(numbers)  # 출력: [1, 1, 3, 4, 5, 9]
```

### 2. `sorted()` 함수 (새로운 정렬된 리스트 반환)
원본 리스트는 그대로 두고, 정렬된 **새로운 리스트**를 얻고 싶을 때 사용합니다.

```python
numbers = [3, 1, 4, 1, 5, 9]
new_numbers = sorted(numbers)
print(new_numbers)  # 출력: [1, 1, 3, 4, 5, 9]
print(numbers)      # 출력: [3, 1, 4, 1, 5, 9] (원본 유지)
```

---

### 💡 유용한 옵션들

#### 1) 내림차순 정렬 (`reverse=True`)
기본값은 오름차순(작은 순서)이지만, `reverse=True`를 추가하면 내림차순(큰 순서)으로 정렬됩니다.
```python
numbers = [1, 2, 3, 4, 5]
numbers.sort(reverse=True)
print(numbers)  # 출력: [5, 4, 3, 2, 1]
```

#### 2) 특정 기준(Key)으로 정렬
문자열의 길이나, 튜플의 특정 요소 등을 기준으로 정렬하고 싶을 때 `key` 인자를 사용합니다.
```python
words = ["banana", "apple", "cherry"]

# 단어 길이 순으로 정렬
words.sort(key=len)
print(words)  # 출력: ['apple', 'cherry', 'banana']
```

---

### 요약
*   **원본을 바꿔도

In [7]:

print("\n" + "=" * 50)
print("예제 3: 번역 체인")
print("=" * 50)

translate_prompt = ChatPromptTemplate.from_template(
    "다음 텍스트를 {target_language}로 번역하세요: {text}"
)

translate_chain = translate_prompt | llm | StrOutputParser()
original = "Hello, how are you today?"
translated = translate_chain.invoke({
    "text": original, 
    "target_language": "한국어"
})
print("번역 결과:", translated)

print("\n" + "=" * 50)
print("예제 4: 감정 분석")
print("=" * 50)

emotion_prompt = ChatPromptTemplate.from_template("""
텍스트: {text}
감정을 분석하고 [긍정/부정/중립]과 1-10점수를 매기세요.
""")

emotion_chain = emotion_prompt | llm | StrOutputParser()
test_text = "오늘 프로젝트가 성공적으로 완료되어서 정말 기쁩니다!"
emotion_result = emotion_chain.invoke({"text": test_text})
print("감정 분석:", emotion_result)

print("\n" + "=" * 50)
print("예제 5: 코드 생성")
print("=" * 50)

code_prompt = ChatPromptTemplate.from_template("""
{language}로 {task} 기능을 구현하는 간단한 코드를 작성하세요.
""")

code_chain = code_prompt | llm | StrOutputParser()
code_result = code_chain.invoke({
    "language": "Python",
    "task": "두 숫자의 최대공약수를 구하는"
})
print("생성된 코드:")
print(code_result)



예제 3: 번역 체인
번역 결과: 안녕하세요, 오늘 기분은 어떠신가요?

예제 4: 감정 분석
감정 분석: 제시해주신 텍스트에 대한 감정 분석 결과입니다.

*   **감정:** 긍정
*   **점수:** 10/10

**이유:** '성공적으로 완료', '정말 기쁩니다'와 같은 표현을 통해 성취감과 매우 높은 만족감을 강하게 드러내고 있습니다.

예제 5: 코드 생성
생성된 코드:
Python에서 두 숫자의 최대공약수(GCD, Greatest Common Divisor)를 구하는 가장 간단하고 효율적인 방법은 내장 라이브러리인 `math` 모듈을 사용하는 것입니다.

### 1. `math` 모듈 사용 (권장)
가장 깔끔하고 성능이 좋은 방법입니다.

```python
import math

a = 24
b = 36

gcd = math.gcd(a, b)
print(f"{a}와 {b}의 최대공약수: {gcd}")
```

---

### 2. 유클리드 호제법을 직접 구현
알고리즘 학습을 위해 직접 구현하고 싶다면 **유클리드 호제법**을 사용하는 것이 좋습니다. (큰 수를 작은 수로 나누어 나머지가 0이 될 때까지 반복하는 방식)

```python
def get_gcd(a, b):
    while b > 0:
        a, b = b, a % b
    return a

# 테스트
print(f"24와 36의 최대공약수: {get_gcd(24, 36)}")
```

**코드 설명:**
* `a % b`는 `a`를 `b`로 나눈 나머지입니다.
* `a, b = b, a % b`를 통해 `b`를 새로운 `a`로, 나머지를 새로운 `b`로 계속 갱신합니다.
* `b`가 0이 되는 순간의 `a`값이 최대공약수가 됩니다.

상황에 맞춰 편하신 방법을 선택해서 사용하시면 됩니다!


In [8]:
print("\n" + "=" * 50)
print("예제 6: 창의적 콘텐츠 생성")
print("=" * 50)

# 창의적 생성용 높은 temperature
llm_creative = ChatGoogleGenerativeAI(
    model=GOOGLE_MODEL,
    temperature=0.9
)

creative_prompt = ChatPromptTemplate.from_template(
    "{topic}에 대한 창의적인 {content_type}를 {style} 스타일로 작성하세요."
)

creative_chain = creative_prompt | llm_creative | StrOutputParser()
creative_result = creative_chain.invoke({
    "topic": "미래의 교통수단",
    "content_type": "아이디어",
    "style": "혁신적이고 실현 가능한"
})
print("창의적 아이디어:", creative_result)

print("\n" + "=" * 50)
print("Gemini 모델 옵션")
print("=" * 50)
print("• gemini-1.5-flash: 빠른 응답, 일반 작업")
print("• gemini-1.5-pro: 정확한 분석, 복잡한 추론")
print("• gemini-pro-vision: 이미지 처리 가능")
print("• temperature: 0.1(정확) ~ 0.9(창의적)")
print("=" * 50)


예제 6: 창의적 콘텐츠 생성
창의적 아이디어: 미래의 교통수단은 단순히 '이동'의 수단을 넘어, **'공간의 최적화'와 '환경과의 공존'**이라는 두 가지 핵심 가치를 지향해야 합니다. 이를 실현하기 위한 혁신적이면서도 기술적으로 실현 가능한 세 가지 아이디어를 제안합니다.

---

### 1. 모듈형 하이브리드 포드 시스템 (Modular Hybrid Pod System)
**[개념]**
개인용 이동 수단(Pod)이 상황에 따라 도로 주행, 자기부상 열차, 혹은 대중교통 플랫폼에 결합하는 방식입니다.

*   **작동 원리:** 출근 시 집 앞까지 온 소형 자율주행 포드에 탑승합니다. 포드는 고속도로 진입 시 자기부상 레일이 깔린 '매크로 캐리어(Macro-carrier)'에 자석으로 결합합니다. 마치 열차의 한 칸처럼 연결되어 고속으로 이동하다가, 목적지 근처에서 분리되어 다시 개별 도로로 주행합니다.
*   **혁신성:** 고속 이동의 효율성과 '문 앞까지(Door-to-Door)'의 편리함을 동시에 충족합니다. 1인용 차량이 고속도로를 점유하는 비효율성을 해소합니다.
*   **실현 가능성:** 현재의 모듈형 자율주행 기술과 하이퍼루프의 자기부상 기술을 결합하면 20~30년 내 충분히 구현 가능합니다.

### 2. 도시 항공 모빌리티를 위한 ‘수직 빌딩 항구’ (Vertical Air-Port Hubs)
**[개념]**
기존의 활주로 중심 공항 개념을 탈피하여, 고층 빌딩의 중간 층을 직접 연결하는 도심형 수직 이착륙(eVTOL) 네트워크입니다.

*   **작동 원리:** 도시 곳곳의 마천루 중간 층에 '드론 스테이션'을 설치합니다. 빌딩 외벽에 자기장으로 고정되는 착륙 플랫폼이 있어, 사용자는 사무실이나 집에서 곧바로 항공 택시에 탑승합니다. 기체는 빌딩 사이사이를 연결하는 지정된 3차원 공중 항로(Sky-lane)를 따라 이동합니다.
*   **혁신성:** 지상 교통 체증을 완벽히 우회합니다. 건물의 고층부를 직접 연결함으로써 도심 이동 시

#### GoogleGenerativeAIEmbeddings

임베딩(Embedding)은 문자열을 **의미를 담은 숫자 벡터**로 바꾸는 것입니다. 벡터끼리의 거리(유사도)로 문장의 의미가 얼마나 가까운지 알 수 있습니다.

- `embed_documents(texts)`: 문자열 리스트 → 벡터 리스트 (입력 순서 유지)
- `embed_query(query)`: 질문 1개 → 벡터 1개
- `gemini-embedding-001`은 모든 문자열을 **3072차원** 벡터로 변환합니다. (Upstage `solar-embedding-1-large`는 4096차원)
- Upstage 예제와 **같은 문장**을 사용하므로 두 모델의 유사도 값을 비교해 볼 수 있습니다.
- 사용 가능한 임베딩 모델은 `client.models.list()`에서 `supported_actions`에 `embedContent`가 있는 모델로 확인합니다.

In [9]:
import math
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 임베딩 모델 생성 (gemini-embedding-001: 문자열 1개를 3072개의 숫자로 변환)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

# 임베딩할 문자열(문서)과 질문(쿼리) - Upstage 예제와 동일한 문장 사용
texts = ["Sung is a professor.", "This is another document"]
query = "What does Sung do?"

# embed_documents(): 문자열 리스트 → 벡터 리스트 (입력 순서와 같은 순서로 반환)
doc_vectors = embeddings.embed_documents(texts)

# embed_query(): 문자열 1개 → 벡터 1개 (검색용 질문을 임베딩할 때 사용)
query_vector = embeddings.embed_query(query)


def cosine_similarity(a, b):
    """두 벡터의 코사인 유사도를 반환합니다. (-1 ~ 1, 1에 가까울수록 의미가 비슷함)"""
    dot = sum(x * y for x, y in zip(a, b))        # 내적
    norm_a = math.sqrt(sum(x * x for x in a))     # 벡터 a의 길이
    norm_b = math.sqrt(sum(y * y for y in b))     # 벡터 b의 길이
    return dot / (norm_a * norm_b)


# 1) 문자열 ↔ 벡터 대응 확인 (앞 5개 값만 표시)
print("■ 1) 문자열 → 벡터 (앞 5개 값만 표시)")
for text, vec in zip(texts + [query], doc_vectors + [query_vector]):
    head = ", ".join(f"{v:+.4f}" for v in vec[:5])
    print(f"   {text!r:<28} → {len(vec)}차원 [{head}, ...]")

# 2) 의미 확인: 질문과 각 문서의 코사인 유사도 (높은 순 정렬)
print(f"\n■ 2) 질문 {query!r} 와의 코사인 유사도")
scores = [(text, cosine_similarity(query_vector, vec)) for text, vec in zip(texts, doc_vectors)]
for text, score in sorted(scores, key=lambda x: x[1], reverse=True):
    print(f"   {score:.4f} {'█' * int(score * 40):<40} {text}")

best_text = max(scores, key=lambda x: x[1])[0]
print(f"\n   → 질문과 의미가 가장 가까운 문장: {best_text!r}")

■ 1) 문자열 → 벡터 (앞 5개 값만 표시)
   'Sung is a professor.'       → 3072차원 [+0.0063, +0.0233, +0.0160, -0.0852, +0.0145, ...]
   'This is another document'   → 3072차원 [-0.0060, -0.0022, +0.0188, -0.0904, -0.0009, ...]
   'What does Sung do?'         → 3072차원 [+0.0148, +0.0161, -0.0031, -0.0908, +0.0020, ...]

■ 2) 질문 'What does Sung do?' 와의 코사인 유사도
   0.8168 ████████████████████████████████         Sung is a professor.
   0.5991 ███████████████████████                  This is another document

   → 질문과 의미가 가장 가까운 문장: 'Sung is a professor.'


#### PDF 직접 읽기 (멀티모달 입력)

Gemini는 PDF를 **별도 파서 없이 모델에 직접 입력**할 수 있습니다. Upstage의 `UpstageDocumentParseLoader`처럼 문서를 텍스트로 먼저 변환하지 않고, 파일을 메시지에 담아 질문하면 모델이 레이아웃(표, 그림 포함)까지 보고 답합니다.

**동작 흐름**: PDF 파일 → base64 인코딩 → `HumanMessage`(text + file) → Gemini → 답변

| 구분 | UpstageDocumentParseLoader | Gemini PDF 입력 |
|---|---|---|
| 결과물 | `Document` 리스트(텍스트/markdown) | 질문에 대한 답변 |
| 용도 | RAG 전처리(분할·임베딩·저장) | 문서 요약, 질의응답, 표 추출 |
| 재사용 | 파싱 결과를 저장해 재사용 | 질문할 때마다 PDF를 다시 전송 |

> PDF를 매번 전송하므로 토큰 비용이 발생합니다. 큰 문서를 반복 질문할 때는 RAG(파싱 → 임베딩 → 검색)가 더 효율적입니다.

In [10]:
import base64
from pathlib import Path
from langchain_core.messages import HumanMessage

# 읽을 PDF 경로 (노트북 기준 상대경로)
PDF_PATH = Path("../data/tutorial-korean.pdf")
PDF_QUESTION = "이 PDF의 핵심 내용을 3줄로 요약해 주세요."

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH.resolve()}")

# PDF 바이트를 base64 문자열로 변환 (메시지에 파일을 직접 담기 위함)
pdf_b64 = base64.b64encode(PDF_PATH.read_bytes()).decode()

# 질문(text)과 PDF(file)를 한 메시지에 함께 전달
message = HumanMessage(
    content=[
        {"type": "text", "text": PDF_QUESTION},
        {"type": "file", "base64": pdf_b64, "mime_type": "application/pdf"},
    ]
)

# PDF 질의응답용 모델 (Gemini API 호출 발생)
pdf_llm = ChatGoogleGenerativeAI(model=GOOGLE_MODEL, temperature=0)
response = pdf_llm.invoke([message])

print(f"파일: {PDF_PATH.name} ({PDF_PATH.stat().st_size / 1024:.0f}KB)")
print(f"질문: {PDF_QUESTION}\n")
print(response.content)

파일: tutorial-korean.pdf (952KB)
질문: 이 PDF의 핵심 내용을 3줄로 요약해 주세요.

[{'type': 'text', 'text': "제공해주신 BlueJ 튜토리얼 PDF의 핵심 내용을 3줄로 요약해 드립니다.\n\n1. 이 문서는 초급 자바 프로그래밍 교육을 위해 설계된 개발 환경인 'BlueJ'의 설치부터 프로젝트 생성, 클래스 편집 및 컴파일 방법을 설명합니다.\n2. 객체 생성, 메소드 실행, 코드패드 활용, 디버깅 등 BlueJ의 주요 기능을 단계별로 안내하여 사용자가 자바 개발 환경에 쉽게 익숙해지도록 돕습니다.\n3. 애플릿 생성, 독립형 어플리케이션 배포, 외부 라이브러리 활용 등 실무적인 추가 기능과 함께 빠른 참조를 위한 요약 정리를 제공합니다.", 'extras': {'signature': 'EnMKcQFpFH0TdmUUD+iANeOkpLFnfK/UXt35B6QBeq5QcxK7ohpXzOEoKUff5a0ieY5YoQMEBGOkcxfUai6QGWx0Qiv/sSssDFjF7dgjm42s3xbdL7VOh5OHfQo2pCc98tCnNeVvWcQ9dz5f4vspalxG6mHY'}}]
